## Multi Modal News Analyzer

### Part 1
#### Data Collection and Preparation
- Text Cleaning
- Tokenization
- Stop Word Removal
- Lemmatization

In [1]:
import pandas as pd
import torch

In [2]:
df=pd.read_json('data/News_Category_Dataset_v3.json', lines=True)

In [3]:
print("\n First 5 rows of the data:")
print(df.head())

print("\n DataFrame info")
print(df.info())


 First 5 rows of the data:
                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss  

In [4]:
#Check for missing values

print("\n Missing values per column:")
print(df.isnull().sum())

#Check distribution of categories

print(df['category'].value_counts())

print("\n Example headlines and descriptions:")
print(df[['headline', 'short_description', 'category']].sample(5))


 Missing values per column:
link                 0
headline             0
category             0
short_description    0
authors              0
date                 0
dtype: int64
category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        

In [5]:
#Function to clean the text
import re
from nltk.corpus import stopwords
import string
stop_words = set(stopwords.words('english'))
def clean_text(text):
    #convert to lowercase
    text=text.lower()

    #remove URLs
    text= re.sub(r'https?:\S*|www\.S*','',text)

    #remove punctuation and special characters
    text=text.translate(str.maketrans('','', string.punctuation))

    #remove stop words
    words=text.split()
    filtered_words=[word for word in words if word not in stop_words]
    # join the words back into a string
    text=" ".join(filtered_words)

    return text
    

In [6]:
df['combined_text'] = df['headline'] + " " + df['short_description']

In [7]:
df['combined_text']=df['combined_text'].apply(clean_text)

In [8]:
#function for lemmatising text with nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ  # Adjective
    elif tag.startswith('V'):
        return wordnet.VERB  # Verb
    elif tag.startswith('N'):
        return wordnet.NOUN  # Noun
    elif tag.startswith('R'):
        return wordnet.ADV  # Adverb
    else:
        return wordnet.NOUN # Default to Noun

def tokenize_and_lemmatize(text):
    tokens = word_tokenize(text)
    tagged_words = nltk.pos_tag(tokens)
    lemmatized_words = []
    for word, tag in tagged_words:
        w_pos = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, pos=w_pos)
        lemmatized_words.append(lemma)
    
    return " ".join(lemmatized_words)
    


In [9]:
import nltk
#nltk.download('wordnet')
#nltk.download('punkt_tab')
#nltk.download('averaged_perceptron_tagger')

In [10]:
df['lemmatized_combined_text']=df['combined_text'].apply(tokenize_and_lemmatize)

In [11]:
print(df['lemmatized_combined_text'].head())

0    4 million american roll sleeve omicrontargeted...
1    american airline flyer charge ban life punch f...
2    23 funny tweet cat dog week sept 1723 dog dont...
3    funny tweet parent week sept 1723 accidentally...
4    woman call cop black birdwatcher lose lawsuit ...
Name: lemmatized_combined_text, dtype: object


In [12]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 8 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   link                      209527 non-null  object        
 1   headline                  209527 non-null  object        
 2   category                  209527 non-null  object        
 3   short_description         209527 non-null  object        
 4   authors                   209527 non-null  object        
 5   date                      209527 non-null  datetime64[ns]
 6   combined_text             209527 non-null  object        
 7   lemmatized_combined_text  209527 non-null  object        
dtypes: datetime64[ns](1), object(7)
memory usage: 12.8+ MB
None


Topic Classification

In [13]:
from sklearn.preprocessing import LabelEncoder

le= LabelEncoder()

# Fit the encoder to your categories and transform them
df['category_id'] = le.fit_transform(df['category'])

# Verify the number of classes and an example mapping
num_classes = len(le.classes_)
print(f"Total Number of Classes: {num_classes}")
print(f"Category IDs (e.g.): {df[['category', 'category_id']].drop_duplicates().head()}")


Total Number of Classes: 42
Category IDs (e.g.):          category  category_id
0       U.S. NEWS           35
2          COMEDY            5
3       PARENTING           22
7      WORLD NEWS           40
8  CULTURE & ARTS            7


In [17]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained('bert-base-uncased')

def encode_texts(texts, max_length=128):
    """ Tokenizes and encodes a list of texts using the Transformer tokenizer. """
    text_list = [str(t) for t in texts.tolist()]
    
    return tokenizer.batch_encode_plus(
        texts.tolist(), 
        max_length=max_length, 
        padding='max_length', 
        truncation=True, 
        return_attention_mask=True,
        return_token_type_ids=False,
        return_tensors='pt'
    )
encoded_data= encode_texts(df['lemmatized_combined_text'])

input_ids= encoded_data['input_ids']
attention_masks=encoded_data['attention_mask']
labels=torch.tensor(df['category_id'].values)

print(f"\nShape of Input IDs: {input_ids.shape}")
print(f"Shape of Attention Masks: {attention_masks.shape}")
print(f"Shape of Labels: {labels.shape}")


Shape of Input IDs: torch.Size([209527, 128])
Shape of Attention Masks: torch.Size([209527, 128])
Shape of Labels: torch.Size([209527])


Split Data and Create DataLoaders

In [20]:
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.model_selection import train_test_split

dataset= TensorDataset(input_ids, attention_masks, labels)

#determine the size of the training set

train_size= int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print( "Training set size: " , len(train_dataset))
print( "Validation set size: ", len(val_dataset))

batch_size = 32

train_dataloader = DataLoader(
    train_dataset,
    sampler=torch.utils.data.RandomSampler(train_dataset),
    batch_size=batch_size
)

validation_dataloader = DataLoader(
    val_dataset,
    sampler=torch.utils.data.SequentialSampler(val_dataset),
    batch_size=batch_size
)

print("\nDataLoaders successfully created. Ready for model setup!")

Training set size:  167621
Validation set size:  41906

DataLoaders successfully created. Ready for model setup!
